In [1]:
import kagglehub
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print(path)  # folder where the files were saved


C:\Users\akul9\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1


In [3]:
df = pd.read_csv(f"{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Exploratory Data Analysis

Let us try to do the string change for each column name and str values

In [5]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

strings = list(df.columns[df.dtypes == "str"])

for s in strings:
    df[s] = df[s].str.lower().str.replace(" ", "_").str.replace("-", "_")
    

Now we have to convert strings into categorical variables, especially the primary outcome variable

In [6]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [7]:
#Clearly totalcharges is a float/number, but it is showing as a string 
df.totalcharges = pd.to_numeric(df.totalcharges, errors = "coerce") #Coerce replaces non-numberic with NaNs 
df.totalcharges = df.totalcharges.fillna(0)

In [8]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges        float64
churn                   str
dtype: object

In [9]:
df.churn = (df.churn == "yes").astype(int) # One way to do this, but we can also use pd.to_numeric()
df.churn

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: churn, Length: 7043, dtype: int64

In [10]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges        float64
churn                 int64
dtype: object

Ok great, now we will use scikitlearn to split our data into train test and validation

In [11]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1) #Easy to get the 80-20 split

In [12]:
#Next we split full train into train and validate. The final split should be 60 - 20 - 20. So we need to get 80 (which is now 100%) into 60 and 20. So what is 20% of 80%? It is 25%

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1) 

len(df_train), len(df_val), len(df_test) #Check that the number of rows are equal

(4225, 1409, 1409)

In [13]:
df_train = df_train.reset_index()
df_val   = df_val.reset_index()
df_test  = df_test.reset_index()

In [14]:
y_train = df_train.churn.values
del df_train['churn']

y_val   = df_val.churn.values
del df_val['churn']

y_test  = df_test.churn.values
del df_test['churn']

Now we are going to use full_train to explore the outcome variable a bit more

In [15]:
df_full_train = df_full_train.reset_index()

In [16]:
df_full_train.churn.value_counts(normalize = True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [17]:
global_churn_rate = df_full_train.churn.mean()

In [18]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv',
               'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod']

Ok now we are going to assess feature importance for both categorical and continuous data. First we will look at categorical data. 

We are going to look at the churn rate within each group

Churn rate is effectively percentage of people churning (leaving to join another company)

In [19]:
df_group = df_full_train.groupby('gender').churn.agg(['mean', 'count'])


Two important metrics of feature importance

1. Difference between the global churn rate and group-specific churn rate
2. Risk ratio is the ration between the group-specific churn rate and the global churn rate

In [20]:
for c in categorical:
    df_group = df_full_train.groupby(c).churn.agg(['mean','count'])
    df_group['diff'] = df_group['mean'] - global_churn_rate
    df_group['risk'] = df_group['mean']/global_churn_rate
    display(df_group)
    print()



,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208


,mean,count,diff,risk
partner,,,,
no,0.329809,2932,0.059841,1.221659
yes,0.205033,2702,-0.064935,0.759472


,mean,count,diff,risk
dependents,,,,
no,0.313760,3968,0.043792,1.162212
yes,0.165666,1666,-0.104302,0.613651


,mean,count,diff,risk
phoneservice,,,,
no,0.241316,547,-0.028652,0.893870
yes,0.273049,5087,0.003081,1.011412


,mean,count,diff,risk
multiplelines,,,,
no,0.257407,2700,-0.012561,0.953474
no_phone_service,0.241316,547,-0.028652,0.893870
yes,0.290742,2387,0.020773,1.076948


... (6 outputs left out)

,mean,count,diff,risk
streamingmovies,,,,
no,0.338906,2213,0.068938,1.255358
no_internet_service,0.077805,1221,-0.192163,0.288201
yes,0.307273,2200,0.037305,1.138182


,mean,count,diff,risk
contract,,,,
month_to_month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


Effectively, when some groups have greater/lesser churn rate than the global churn rate, we know that such groups will be important for the machine learning models. Those features will have the most information for predicting whether a new person is going to churn or not. 

One metric that can be important to determine the importance of a feature is Mutual Information. Mutual Information is tells us how much we can learn about the churn rate if we know about a certain feature (such as if they have a month-to-month contract) 

In [21]:
#Use sklearn for mutual information

from sklearn.metrics import mutual_info_score

def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

mi_score = df_full_train[categorical].apply(mutual_info_churn_score)    

mi_score.sort_values(ascending=False)


contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

Ok now we can see how to measure feature importance for continous/numerical variables

In [22]:
df_full_train[numerical].corrwith(df_full_train.churn) #Gives us the correlation term

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

Ok so we can look at which features are important and which features are not that important - I am not sure what exactly we do with that information... BUT what we will do is look at ONE HOT ENCODING.

One hot encoding is turning each level of a categorical variable into a logical array, and storing it into the data frame. By doing so, we can prepare our feature matrix for the machine learning model 

In [23]:
from sklearn.feature_extraction import DictVectorizer

train_dicts = df_train[numerical + categorical].to_dict(orient='records')

In [24]:
dv = DictVectorizer(sparse=False) 

In [25]:
dv.fit(train_dicts) #Now learn the parameters and features from the dictionary

DictVectorizer(sparse=False)

In [26]:
X_train = dv.fit_transform(train_dicts)

In [27]:
#Now we do the same for validation dictionaries

val_dicts = df_val[numerical + categorical].to_dict(orient='records')
X_val     = dv.transform(val_dicts) #We do not fit on the validation dataset

Now that we have prepared our feature matrix, we shall next fit our logistic regression model.

The difference between linear and logistic regression is that we add a "sigmoid" function to the linear equation, such that our output is a probability between 0 and 1. Usually 0.5 indicates churn is as likely as no churn. Closer to 1 indicates churn, while closer to 0 indicates no churn

In [28]:
#Use sci-kitlearn to fit logistic regression

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train,y_train)

c:\Users\akul9\Courses\ml-zoomcamp\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [29]:
#Let us investigate the weights

w0 = model.intercept_[0]
w  = model.coef_[0].round(3)

w0, w

(np.float64(-0.10910575411731115),
 array([ 0.476, -0.175, -0.408, -0.03 , -0.078,  0.063, -0.089, -0.081,
        -0.034, -0.073, -0.336,  0.317, -0.089,  0.004, -0.258,  0.142,
         0.009,  0.063, -0.089, -0.081,  0.266, -0.089, -0.284, -0.231,
         0.123, -0.166,  0.059, -0.087, -0.032,  0.07 , -0.059,  0.142,
        -0.25 ,  0.216, -0.121, -0.089,  0.102, -0.071, -0.089,  0.052,
         0.213, -0.089, -0.232, -0.07 ,  0.   ]))

In [30]:
# Now let us try to predict using this trained model

model.predict(X_val) #Hard predictions - each row gets assigned a 0 or 1
y_pred = model.predict_proba(X_val)[:,1] #Gives probability that churn is 0 (first column) or 1 (second column)


In [31]:
churn_decision = (y_pred >= 0.5) #We decide that anyone above 0.5 as the probability will churn. Note that this is the probability that the value of churn is "1"

df_val[churn_decision].customerid

3       8433_wxgna
8       3440_jpscl
11      2637_fkfsy
12      7228_omtpn
19      6711_fldfb
           ...    
1397    5976_jcjrh
1398    2034_cgrhz
1399    5276_kqwhg
1407    6521_yytyi
1408    3049_solay
Name: customerid, Length: 312, dtype: str

Ok we can now measure the performance of the model. So we will calculate accuracy - how many correct predictions?

In [32]:
(churn_decision == y_val).mean() #We check our churn decision and check with true churners

np.float64(0.8026969481902059)

Ok let us see how we can use the model. We want to use the full training dataset (train + val) because we got quite a good prediction (0.8). So we fit the model for full train. Then we will use that model to predict churn rate in the test data set, and then compute the classification accuracy by comparing the predicted churn rate with the actual churn rate 

In [33]:
#Step 0. Get the true churn rate from the full training dataset and delete that column
y_full_train = df_full_train.churn.values

#del df_full_train['churn']

In [46]:
#Step 1. Get the full dataset in dictionary form for the model fitting and prediction

full_train_dicts = df_full_train[numerical + categorical].to_dict(orient="records")
test_dicts       = df_test[numerical + categorical].to_dict(orient="records")
#Step 2. Convert dictionaries into the feature matrices
dv = DictVectorizer(sparse=False) 
X_full_train = dv.fit_transform(full_train_dicts)
X_test       = dv.transform(test_dicts)
#Step 3. Use feature matrix to fit the model

model.fit(X_full_train, y_full_train)

c:\Users\akul9\Courses\ml-zoomcamp\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [47]:
#Step 4. Get the model accuracy using the test dataset

y_pred = model.predict_proba(X_test)[:,1]
churn_decision = (y_pred >= 0.5)

classification_acc = (churn_decision == y_test).mean()
classification_acc

np.float64(0.8140525195173882)

Let us try and use this model to predict churn rate for a particular person

In [75]:
a = test_dicts[199]
X_individual = dv.transform(a)
model.predict_proba(X_individual)[:,1] > 0.5

array([ True])

In [76]:
df

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590_vhveg,female,0,yes,no,1,no,no_phone_service,dsl,no,...,no,no,no,no,month_to_month,yes,electronic_check,29.85,29.85,0
1,5575_gnvde,male,0,no,no,34,yes,no,dsl,yes,...,yes,no,no,no,one_year,no,mailed_check,56.95,1889.50,0
2,3668_qpybk,male,0,no,no,2,yes,no,dsl,yes,...,no,no,no,no,month_to_month,yes,mailed_check,53.85,108.15,1
3,7795_cfocw,male,0,no,no,45,no,no_phone_service,dsl,yes,...,yes,yes,no,no,one_year,no,bank_transfer_(automatic),42.30,1840.75,0
4,9237_hqitu,female,0,no,no,2,yes,no,fiber_optic,no,...,no,no,no,no,month_to_month,yes,electronic_check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840_resvb,male,0,yes,yes,24,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,yes,mailed_check,84.80,1990.50,0
7039,2234_xaduh,female,0,yes,yes,72,yes,yes,fiber_optic,no,...,yes,no,yes,yes,one_year,yes,credit_card_(automatic),103.20,7362.90,0
7040,4801_jzazl,female,0,yes,yes,11,no,no_phone_service,dsl,yes,...,no,no,no,no,month_to_month,yes,electronic_check,29.60,346.45,0
7041,8361_ltmkd,male,1,yes,no,4,yes,yes,fiber_optic,no,...,no,no,no,no,month_to_month,yes,mailed_check,74.40,306.60,1
